<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/knowledge_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/knowledge_distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series - Example: Knowledge Distillation
![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)
This notebook demonstrates how to use [OptiPFair](https://github.com/peremartra/optipfair) to recover performance after depth pruning using knowledge distillation.  
It follows the full recommended workflow with public APIs only: load a teacher model, create a depth-pruned student, distill knowledge, and visualize training curves.

The benchmark stage with lm_eval is intentionally excluded in this example.

##Recommended Environment

- **Platform**: [Google Colab](https://colab.research.google.com)  
- **Hardware**: GPU runtime (recommended: T4 or better for 0.8B-1B models)  
- **Dependencies**: Installed in Section 0

##by Pere Martra.

- [LinkedIn](https://www.linkedin.com/in/pere-martra)  
- [GitHub](https://github.com/peremartra)  
- [X / Twitter](https://x.com/peremartra)

---

> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!

---
If you want your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.

# Knowledge Distillation Example

This notebook demonstrates how to recover a depth-pruned model with OptiPFair knowledge distillation.
We follow the recommended sequence for post-pruning recovery:
1. Load a teacher model
2. Prepare a small recovery dataset
3. Build a depth-pruned student with OptiPFair
4. Recover performance with knowledge distillation using `opf.distill_model()`
5. Plot training losses from `stats['loss_history']`

## 0. Environment and Dependencies\n
First, install the required libraries and define the base configuration.

In [1]:
!pip install git+https://github.com/peremartra/optipfair.git

  Cloning https://github.com/peremartra/optipfair.git to /tmp/pip-req-build-a_ciili6
  Running command git clone --filter=blob:none --quiet https://github.com/peremartra/optipfair.git /tmp/pip-req-build-a_ciili6
  Resolved https://github.com/peremartra/optipfair.git to commit 749825cfff42fd2e5213c0309087392532243af2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for optipfair: filename=optipfair-0.3.0-py3-none-any.whl size=63427 sha256=518a46079bcca280cb1d1f79419f1440222cbd9423b3d823e5809c6aca7c07d7
  Stored in directory: /tmp/pip-ephem-wheel-cache-qf6ehtn2/wheels/34/ee/95/45c0e77756d6cde346debec33bab0826fe994bc1baeeda4f31
Successfully built optipfair


In [2]:
!pip install -q transformers==5.4.0
!pip install -q datasets tqdm matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 76.4 MB/s eta 0:00:00


In [3]:
RECOVERY_SAMPLES = 2000   # Use a small number for the example\n
EPOCHS = 3
LEARNING_RATE = 4e-5
BATCH_SIZE = 4
MAX_LENGTH = 512
LAYERS_TO_REMOVE_COUNT = 2

In [4]:
import torch
from copy import deepcopy
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset, Dataset
from torch.utils.data import TensorDataset, DataLoader, random_split
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import optipfair as opf

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Teacher Model
Load the teacher model in evaluation mode and freeze all its parameters.

The teacher is only used as a supervision signal during distillation.

In [5]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"   # or "google/gemma-3-270m" for a smaller option\n

print(f"Loading Teacher model: {MODEL_NAME}")
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)

teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_teacher_layers = teacher_model.config.num_hidden_layers
print(f"Teacher: {n_teacher_layers} layers, {teacher_model.num_parameters():,} params")

Loading Teacher model: Qwen/Qwen3.5-0.8B-Base


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

Teacher: 24 layers, 752,393,024 params


## 2. Prepare Training Dataset
Adapted from NB03: load Cosmopedia in streaming mode, tokenize text, and create an 80/20 train/validation split.

In [6]:
print("Loading Cosmopedia dataset...")
dataset_name = "HuggingFaceTB/cosmopedia"
subsets = ["stories", "wikihow", "openstax", "web_samples_v1"]
samples_per_subset = int(RECOVERY_SAMPLES / len(subsets))\

all_samples = []
for subset in subsets:
    print(f"  Loading {subset}...")
    subset_data = load_dataset(dataset_name, subset, split="train", streaming=True)
    subset_samples = list(subset_data.take(samples_per_subset))
    all_samples.extend(subset_samples)
    print(f"    Collected {len(subset_samples):,} samples")

distillation_dataset = Dataset.from_dict({"text": [s["text"] for s in all_samples]})
print(f"Total samples: {len(distillation_dataset):,}")

Loading Cosmopedia dataset...
  Loading stories...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/43 [00:00<?, ?it/s]

    Collected 500 samples
  Loading wikihow...


...

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

    Collected 500 samples
  Loading web_samples_v1...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/139 [00:00<?, ?it/s]

    Collected 500 samples
Total samples: 2,000


In [7]:
print("Tokenizing...")
texts = [item["text"] for item in distillation_dataset]
tokenized_data = []
for i in tqdm(range(0, len(texts), 100), desc="Tokenizing"):
    batch = tokenizer(
        texts[i:i + 100],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    tokenized_data.append(batch)

input_ids = torch.cat([b["input_ids"] for b in tokenized_data], dim=0)
attention_mask = torch.cat([b["attention_mask"] for b in tokenized_data], dim=0)
full_dataset = TensorDataset(input_ids, attention_mask)

generator = torch.Generator().manual_seed(42)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], generator=generator
)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Train: {len(train_dataset):,} samples ({len(train_dataloader):,} batches)")
print(f"Val:   {len(val_dataset):,} samples")

Tokenizing...


Tokenizing:   0%|          | 0/20 [00:00<?, ?it/s]

Train: 1,600 samples (400 batches)
Val:   400 samples


## 3. Create Pruned Student Model\n
Analyze layer importance with calibration data, remove the least important layers, and prepare the student for training.

In [8]:
print("Analyzing layer importance...")
student_model = deepcopy(teacher_model)
importance_scores = opf.analyze_layer_importance(
    student_model,
    train_dataloader,
    show_progress=True
)

print("\nLayer importance scores (lower = less important):")
for layer_idx, score in sorted(importance_scores.items()):
    print(f"  Layer {layer_idx:2d}: {score:.6f}")

Analyzing layer importance...


Processing batches: 100%|██████████| 400/400 [11:18<00:00,  1.70s/it]


Layer importance scores (lower = less important):
  Layer  0: 0.840756
  Layer  1: 0.168809
  Layer  2: 0.200107
  Layer  3: 0.173740
  Layer  4: 0.140527
  Layer  5: 0.123701
  Layer  6: 0.137432
  Layer  7: 0.116289
  Layer  8: 0.074131
  Layer  9: 0.062617
  Layer 10: 0.076602
  Layer 11: 0.095449
  Layer 12: 0.078008
  Layer 13: 0.084619
  Layer 14: 0.119063
  Layer 15: 0.144600
  Layer 16: 0.093105
  Layer 17: 0.073428
  Layer 18: 0.080742
  Layer 19: 0.079346
  Layer 20: 0.059014
  Layer 21: 0.045449
  Layer 22: 0.072061
  Layer 23: 0.273174


In [9]:
LAYERS_TO_REMOVE = sorted(
    importance_scores.keys(),
    key=lambda x: importance_scores[x]
)[:LAYERS_TO_REMOVE_COUNT]

print(f"Layers selected for removal: {LAYERS_TO_REMOVE}")

Layers selected for removal: [21, 20]


In [10]:
student_model = opf.prune_model_depth(
    model=student_model,
    layer_indices=LAYERS_TO_REMOVE,
    show_progress=True
)

for param in student_model.parameters():
    param.requires_grad = True

n_student_layers = student_model.config.num_hidden_layers
print(f"\nTeacher layers: {n_teacher_layers}")
print(f"Student layers: {n_student_layers} (removed {LAYERS_TO_REMOVE})")
print(f"Student params: {student_model.num_parameters():,}")

Removing layers: 100%|██████████| 24/24 [00:00<00:00, 293479.00it/s]


Teacher layers: 24
Student layers: 22 (removed [21, 20])
Student params: 709,282,304


## 4. Knowledge Distillation with OptiPFair
Run labels-only distillation (hard labels + skew KLD) using the OptiPFair public API.

Feature alignment is disabled in this configuration (gamma=0, delta=0).

In [11]:
student_to_train = deepcopy(student_model)

trained_student, stats = opf.distill_model(
    student_model=student_to_train,
    teacher_model=teacher_model,
    dataloader=train_dataloader,
    alpha=0.6,
    beta=0.4,
    gamma=0.0,
    delta=0.0,
    temperature=2.0,
    skew_alpha=0.4,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    accumulation_steps=4,
    show_progress=True,
    return_stats=True,
)

print("\nTraining complete")
print(f"  Total time:        {stats['total_time_seconds']:.1f}s ({stats['total_time_seconds'] / 60:.1f} min)")
print(f"  Avg time/epoch:    {stats['avg_time_per_epoch']:.1f}s")
print(f"  Final total loss:  {stats['loss_history']['total'][-1]:.4f}")
print(f"  Final task loss:   {stats['loss_history']['task'][-1]:.4f}")
print(f"  Final logits loss: {stats['loss_history']['logits'][-1]:.4f}")

Epoch 1/3:   0%|          | 0/400 [00:00<?, ?it/s]

IndexError: list index out of range

## 5. Visualize Training Loss Curves
Plot total, task, and logits losses using the stats dictionary returned by distill_model.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

loss_keys = [
    ("total",  "Total Loss"),
    ("task",   "Task Loss (Cross-Entropy)"),
    ("logits", "Logits Loss (Skew KLD)"),
]

for idx, (key, title) in enumerate(loss_keys):
    axes[idx].plot(
        stats['loss_history'][key],
        marker='o', linewidth=2, markersize=8
    )
    axes[idx].set_title(title, fontsize=12)
    axes[idx].set_xlabel('Epoch')
    axes[idx].set_ylabel('Loss')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle(
    'Training Progress: Knowledge Distillation (Labels Only)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('kd_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to kd_training_curves.png")

## 6. Optional Save Trained Student
Save the distilled student and tokenizer for later use.

In [ ]:
OUTPUT_PATH = "./kd-student"
trained_student.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f"Trained student saved to {OUTPUT_PATH}")